# 05. Revenue and expenditure decomposition

Move from B.9 levels to the exact identity B = R - E, and decompose each adjacent-year balance change into a revenue change and an expenditure change.

**Reads**

- `data/processed/subsector_accounts_1977_2025.csv`
- `outputs/tables/revenue_expenditure_change_decomposition.csv`
- `outputs/tables/largest_balance_movements.csv`

**Writes**

- Nothing. All three tables are persisted by the pipeline.

**Method reference:** `METHODOLOGY.md` section 7

In [ ]:
"""Notebook environment: locate the repository and expose its data layers."""

import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

# Resolve the repository root from wherever the kernel was started, so the
# notebook works both from the repository root and from the notebooks directory.
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

%matplotlib inline

from portugal_fiscal_balance.analysis import figures

RAW = ROOT / 'data' / 'raw'
INTERIM = ROOT / 'data' / 'interim'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
METRICS = ROOT / 'outputs' / 'metrics'

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

print('repository:', ROOT.name)
print('pipeline outputs present:', (PROCESSED / 'fiscal_balances_1977_2025.csv').exists())

## 1. Levels

For every sector-year with detailed components,

$$B_{i,t} = R_{i,t} - E_{i,t}.$$

Ratios to GDP are used for comparability across five decades of nominal growth.

In [ ]:
accounts = pd.read_csv(PROCESSED / 'subsector_accounts_1977_2025.csv')
changes = pd.read_csv(TABLES / 'revenue_expenditure_change_decomposition.csv')
level_columns = [
    'year',
    'total_revenue_pct_gdp',
    'total_expenditure_pct_gdp',
    'balance_pct_gdp',
    'interest_pct_gdp',
    'gfcf_pct_gdp',
]
central = accounts.loc[accounts['sector'].eq('central_government') & accounts['year'].ge(2015), level_columns]
display(central.round(2))

In [ ]:
figure = figures.revenue_expenditure(accounts, 'central_government')

The line breaks at 1996-1999 because subsector components do not exist for those
years. The break is drawn deliberately: joining 1995 to 2000 would suggest a path
the sources do not contain.

In [ ]:
figure = figures.revenue_expenditure(accounts, 'social_security_funds')

## 2. Changes

Differencing the identity gives

$$\Delta B_{i,t} = \Delta R_{i,t} - \Delta E_{i,t},$$

which holds exactly for adjacent years inside a continuous source block. No
change is computed across the 1995-to-2000 subsector gap; those rows are dropped
rather than bridged.

In [ ]:
ssf_changes = changes.loc[changes['sector'].eq('social_security_funds')]
display(ssf_changes.tail(10).round(1))
print('rows in the table:', len(changes))
print('year gaps present:', sorted(changes['year_gap'].dropna().unique().tolist()))

In [ ]:
last_year = int(changes['year'].max())
figure = figures.revenue_expenditure_changes(
    changes, 'social_security_funds', start_year=2010, end_year=last_year
)

In [ ]:
figure = figures.revenue_expenditure_changes(
    changes, 'general_government', start_year=2001, end_year=last_year
)

## 3. Which episodes were revenue-driven and which expenditure-driven?

The ranked episodes come from notebook 06, which selects them on the GDP-scaled
aggregate change within each regime. Here each is split into the revenue and
expenditure movements of **the subsector that dominates it**, not of the aggregate,
so both halves describe the same entity.

Expenditure enters the balance negatively. The contribution column is therefore minus
the expenditure change, and it is that column which adds to the revenue change to give
the subsector's balance change. The split residual is the gap between the canonical
panel's measure of that subsector change and the account panel's: two source families
that are not forced to agree.

In [ ]:
movements = pd.read_csv(TABLES / 'largest_balance_movements.csv')
display(
    movements[
        [
            'regime',
            'year',
            'dominant_subsector',
            'dominant_subsector_change_m_eur',
            'dominant_revenue_change_m_eur',
            'dominant_expenditure_change_m_eur',
            'dominant_expenditure_contribution_m_eur',
            'dominant_split_error_m_eur',
        ]
    ].round(1)
)

## 4. Decomposition check

The residual is an arithmetic identity check on the persisted table, so anything
other than numerical noise would indicate a defect.

In [ ]:
print('max |decomposition residual| (M EUR):', float(changes['decomposition_error_m_eur'].abs().max()))
display(
    changes.groupby('sector')['decomposition_error_m_eur']
    .apply(lambda column: column.abs().max())
    .rename('max_abs_residual_m_eur')
    .to_frame()
)

## Interpretation limits

1. A revenue or expenditure change is **not decomposed into policy and
   macroeconomic components**. Doing so requires assumptions this repository
   does not make.
2. Totals are used, so a change in revenue may reflect composition shifts that
   are visible only in the component columns of the account panel.
3. Changes are **never computed across a source gap**, which is why the modern
   subsector series begins contributing changes in 2001.

---

[Previous: 04. Long-run balance decomposition](04_balance_decomposition.ipynb) | [Next: 06. Year-to-year balance attribution](06_year_to_year_attribution.ipynb)

Every table shown above is also persisted as CSV, so results can be checked without reading notebook state. To rebuild everything from the bundled raw sources:

```bash
poetry install
make all
```